In [1]:
import sys
import time
import json
from pathlib import Path

import torch

BASE = Path("/workspace/tiny-llm-from-scratch")
sys.path.append(str(BASE))

from tokenizer.char_tokenizer import CharTokenizer
from model.tiny_transformer import TinyGPT, GPTConfig

In [2]:
CHECKPOINT = BASE / "checkpoints/tinyllm_100k_char_best.pt"

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu"
)

print("Checkpoint Loaded")

Checkpoint Loaded


In [3]:
tokenizer = CharTokenizer.load(
    BASE / "tokenizer/tokenizer_char.json"
)

print("Vocabulary Size:", tokenizer.vocab_size)

Vocabulary Size: 44


In [4]:
cfg = GPTConfig(**checkpoint["gpt_config"])

model = TinyGPT(cfg)

model.load_state_dict(checkpoint["model_state"])

model.eval()

print("Model Ready")

Model Ready


In [5]:
print("="*60)

print("Model Name :", checkpoint["config"]["model_name"])
print("Parameters :", checkpoint["param_count"])
print("Checkpoint :", checkpoint["step"])
print("Best Loss :", checkpoint["best_val_loss"])

print("="*60)

Model Name : tinyllm_100k_char
Parameters : 111104
Checkpoint : 4750
Best Loss : 0.05055588111281395


In [6]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total Parameters :", total)
print("Trainable :", trainable)

size_mb = total * 4 / (1024**2)

print(f"Approx Model Size : {size_mb:.2f} MB")

Total Parameters : 111104
Trainable : 111104
Approx Model Size : 0.42 MB


In [7]:
print("="*40)

print("Vocabulary")

print("="*40)

for idx in range(tokenizer.vocab_size):
    print(idx, repr(tokenizer.itos[idx]))

Vocabulary
0 '<UNK>'
1 '\n'
2 ' '
3 ','
4 '-'
5 '.'
6 'A'
7 'D'
8 'F'
9 'G'
10 'I'
11 'L'
12 'M'
13 'P'
14 'Q'
15 'R'
16 'S'
17 'T'
18 'a'
19 'b'
20 'c'
21 'd'
22 'e'
23 'f'
24 'g'
25 'h'
26 'i'
27 'j'
28 'k'
29 'l'
30 'm'
31 'n'
32 'o'
33 'p'
34 'q'
35 'r'
36 's'
37 't'
38 'u'
39 'v'
40 'w'
41 'x'
42 'y'
43 'z'


In [8]:
prompt = "Python"

ids = tokenizer.encode(prompt)

x = torch.tensor([ids])

start = time.time()

with torch.inference_mode():

    output = model.generate(
        x,
        max_new_tokens=100
    )

elapsed = time.time() - start

print(f"Time : {elapsed:.4f} sec")

Time : 0.3228 sec


In [9]:
prompt = "Python"

ids = tokenizer.encode(prompt)

x = torch.tensor([ids])

start = time.time()

with torch.inference_mode():

    output = model.generate(
        x,
        max_new_tokens=100
    )

elapsed = time.time() - start

print(f"Time : {elapsed:.4f} sec")

Time : 0.2231 sec


In [10]:
memory = 0

for p in model.parameters():

    memory += p.numel() * p.element_size()

print(f"Memory : {memory/1024/1024:.2f} MB")

Memory : 0.42 MB


In [11]:
prompts = [
    "Python",
    "Docker",
    "FastAPI",
    "SQL",
    "HTML",
    "Machine Learning",
    "Artificial Intelligence",
    "Django"
]

score = 0

for prompt in prompts:

    ids = tokenizer.encode(prompt)

    x = torch.tensor([ids])

    with torch.inference_mode():

        output = model.generate(
            x,
            max_new_tokens=80,
            temperature=0.8,
            top_k=20
        )

    text = tokenizer.decode(output[0].tolist())

    print("="*60)
    print(prompt)
    print("-"*60)
    print(text)

    if len(text) > len(prompt):
        score += 1

print("\nAccuracy Score:", score, "/", len(prompts))

Python
------------------------------------------------------------
Python web framework used to build web applications.
FastAPI is a Python framework use
Docker
------------------------------------------------------------
Docker is used to package applications and run them inside containers.
A database stor
FastAPI
------------------------------------------------------------
FastAPI is a Python framework used to build APIs.
Docker is used to package application
SQL
------------------------------------------------------------
SQL is a relational database.
RAG means Retrieval Augmented Generation.
RAG helps a
HTML
------------------------------------------------------------
<UNK>TMLL is trained using next-token prediction.

Artificial intelligence is the scienc
Machine Learning
------------------------------------------------------------
Machine Learning models improve by reducing los during trainining.
The training loop includes fo
Artificial Intelligence
------------------------------

In [12]:
temps = [0.2, 0.5, 0.8, 1.0]

for t in temps:

    ids = tokenizer.encode("Python")

    x = torch.tensor([ids])

    with torch.inference_mode():

        out = model.generate(
            x,
            max_new_tokens=60,
            temperature=t
        )

    print("="*60)
    print("Temperature:", t)
    print(tokenizer.decode(out[0].tolist()))

Temperature: 0.2
Python web framework used to build web applications.
FastAPI is a 
Temperature: 0.5
Python web framework used to build web applications.
FastAPI is a 
Temperature: 0.8
Python web framework used to build web applications.
FastAPI is a 
Temperature: 1.0
Python build APIs.
Docker is used to package aplications and run t


In [13]:
for k in [5,10,20,40]:

    ids = tokenizer.encode("Python")

    x = torch.tensor([ids])

    with torch.inference_mode():

        out = model.generate(
            x,
            max_new_tokens=60,
            top_k=k
        )

    print("="*60)
    print("Top K:", k)
    print(tokenizer.decode(out[0].tolist()))

Top K: 5
Python framework used to build web applications.
FastAPI is a Pyth
Top K: 10
Python work used to build web applications.
FastAPI is a Python fr
Top K: 20
Python web framework used to build web applications.
FastAPI is a 
Top K: 40
Python lun web frard pas, loss calculation, backpropagation, and o


In [14]:
tests = [
    "Hello",
    "API",
    "Data",
    "Database",
    "Neural",
    "Linux",
    "Cloud",
    "Programming"
]

for prompt in tests:

    ids = tokenizer.encode(prompt)

    x = torch.tensor([ids])

    with torch.inference_mode():

        out = model.generate(
            x,
            max_new_tokens=60
        )

    print("="*60)
    print(prompt)
    print(tokenizer.decode(out[0].tolist()))

Hello
<UNK>elloatabase.
RAG means Retrieval Augmented Generation.
RAG helps
API
API is used to package applications and run them inside contain
Data
Database stores structured information.
PostgreSQL is a relation
Database
Database stores structured information.
PostgreSQL is a relational d
Neural
<UNK>eural lanswer using external documents.
A vector database stores 
Linux
Linuxt-token prediction.

Artificial intelligence is the science 
Cloud
<UNK>loud APIs.
Docker is used to package applications and run them i
Programming
ProgrammingreSQL is a relational database.
RAG means Retrieval Augmente


In [17]:
print("=" * 60)
print("TinyGPT Benchmark Report")
print("=" * 60)

# -----------------------------
# Parameter Statistics
# -----------------------------
total_params = sum(p.numel() for p in model.parameters())

memory_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

memory_mb = memory_bytes / (1024 ** 2)

# -----------------------------
# Generation Speed Benchmark
# -----------------------------
prompt = "Python"

ids = tokenizer.encode(prompt)

x = torch.tensor([ids])

start = time.time()

with torch.inference_mode():

    output = model.generate(
        x,
        max_new_tokens=100
    )

elapsed = time.time() - start

speed = 100 / elapsed

# -----------------------------
# Prompt Benchmark
# -----------------------------
test_prompts = [
    "Python",
    "Docker",
    "FastAPI",
    "SQL",
    "HTML",
    "Machine Learning",
    "Artificial Intelligence",
    "Django"
]

score = 0

for prompt in test_prompts:

    ids = tokenizer.encode(prompt)

    x = torch.tensor([ids])

    with torch.inference_mode():

        output = model.generate(
            x,
            max_new_tokens=80,
            temperature=0.8,
            top_k=20
        )

    text = tokenizer.decode(output[0].tolist())

    if len(text) > len(prompt):
        score += 1

# -----------------------------
# Report
# -----------------------------
print("Model Name      :", checkpoint["config"]["model_name"])
print("Parameters      :", total_params)
print("Vocabulary      :", tokenizer.vocab_size)
print("Best Loss       :", checkpoint["best_val_loss"])
print(f"Generation Speed: {speed:.2f} tokens/sec")
print(f"Memory Usage    : {memory_mb:.2f} MB")
print(f"Prompt Score    : {score}/{len(test_prompts)}")

print()

if score == len(test_prompts):
    rating = "★★★★★ Excellent"
elif score >= 6:
    rating = "★★★★ Very Good"
elif score >= 4:
    rating = "★★★ Good"
else:
    rating = "★★ Needs Improvement"

print("Overall Rating  :", rating)

print

TinyGPT Benchmark Report
Model Name      : tinyllm_100k_char
Parameters      : 111104
Vocabulary      : 44
Best Loss       : 0.05055588111281395
Generation Speed: 473.14 tokens/sec
Memory Usage    : 0.42 MB
Prompt Score    : 8/8

Overall Rating  : ★★★★★ Excellent


<function print(*args, sep=' ', end='\n', file=None, flush=False)>